<a href="https://colab.research.google.com/github/everestso/AI-Education/blob/main/DarkChambers_DeepLearning_DRuby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "gymnasium[atari,accept-rom-license]" moviepy

In [2]:
%pip install -q "autorom[accept-rom-license]"
!AutoROM --accept-license

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.13/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


In [3]:
import os
import glob
import shutil
import random

import numpy as np
import pandas as pd
import gymnasium as gym
import ale_py

from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

# Register the ALE environments with Gymnasium.
gym.register_envs(ale_py)

In [4]:
env = gym.make(
    "ALE/Darkchambers-v5",
    render_mode="rgb_array",
)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Available actions:", env.unwrapped.get_action_meanings())

env.close()

Observation space: Box(0, 255, (210, 160, 3), uint8)
Action space: Discrete(18)
Available actions: ['NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'DOWN', 'UPRIGHT', 'UPLEFT', 'DOWNRIGHT', 'DOWNLEFT', 'UPFIRE', 'RIGHTFIRE', 'LEFTFIRE', 'DOWNFIRE', 'UPRIGHTFIRE', 'UPLEFTFIRE', 'DOWNRIGHTFIRE', 'DOWNLEFTFIRE']


In [5]:
def choose_action(observation, rng):
    """
    Select an action using a simple state-independent policy.

    Dark Chambers combined movement-and-fire actions:
      10 = UPFIRE
      11 = RIGHTFIRE
      12 = LEFTFIRE
      13 = DOWNFIRE
      14 = UPRIGHTFIRE
      15 = UPLEFTFIRE
      16 = DOWNRIGHTFIRE
      17 = DOWNLEFTFIRE
    """

    useful_actions = [10, 11, 12, 13, 14, 15, 16, 17]
    return int(rng.choice(useful_actions))

In [6]:
video_folder = "darkchambers_video"

if os.path.exists(video_folder):
    shutil.rmtree(video_folder)

# IMPORTANT: Create a fresh, unwrapped environment.
env = gym.make(
    "ALE/Darkchambers-v5",
    render_mode="rgb_array",
)

env = RecordVideo(
    env,
    video_folder=video_folder,
    episode_trigger=lambda episode_number: True,
    name_prefix="darkchambers-random-agent",
    disable_logger=True,
)

seed = 42
rng = np.random.default_rng(seed)
env.action_space.seed(seed)

observation, info = env.reset(seed=seed)

action_names = env.unwrapped.get_action_meanings()
transitions = []
actions_taken = []
total_reward = 0.0

maximum_steps = 2000
steps_per_decision = 5

terminated = False
truncated = False
step_number = 0

while (
    step_number < maximum_steps
    and not terminated
    and not truncated
):
    action = choose_action(observation, rng)
    actions_taken.append(action)

    # Hold the selected action briefly instead of changing direction
    # on every individual environment step.
    for _ in range(steps_per_decision):
        next_observation, reward, terminated, truncated, info = env.step(action)

        total_reward += reward

        # Store compact transition information rather than two complete
        # 210 x 160 x 3 images for every step.
        transitions.append({
            "step": step_number,
            "action_number": action,
            "action": action_names[action],
            "reward": reward,
            "cumulative_reward": total_reward,
            "observation_shape": next_observation.shape,
        })

        observation = next_observation
        step_number += 1

        if terminated or truncated or step_number >= maximum_steps:
            break

env.close()

video_files = sorted(glob.glob(f"{video_folder}/*.mp4"))

if not video_files:
    raise FileNotFoundError("The episode completed, but no MP4 file was created.")

video_path = video_files[0]

print("Created:", video_path)
print("Steps completed:", len(transitions))
print("Total episode reward:", total_reward)
print("Terminated:", terminated)
print("Truncated:", truncated)

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Created: darkchambers_video/darkchambers-random-agent-episode-0.mp4
Steps completed: 2000
Total episode reward: 10.0
Terminated: False
Truncated: False


In [7]:
display(
    Video(
        video_path,
        embed=True,
        width=640,
    )
)

In [8]:
transitions_df = pd.DataFrame(transitions)
transitions_df

,step,action_number,action,reward,cumulative_reward,observation_shape
0,0,10,UPFIRE,0.0,0.0,"(210, 160, 3)"
1,1,10,UPFIRE,0.0,0.0,"(210, 160, 3)"
2,2,10,UPFIRE,0.0,0.0,"(210, 160, 3)"
3,3,10,UPFIRE,0.0,0.0,"(210, 160, 3)"
4,4,10,UPFIRE,0.0,0.0,"(210, 160, 3)"
...,...,...,...,...,...,...
1995,1995,12,LEFTFIRE,0.0,10.0,"(210, 160, 3)"
1996,1996,12,LEFTFIRE,0.0,10.0,"(210, 160, 3)"
1997,1997,12,LEFTFIRE,0.0,10.0,"(210, 160, 3)"
1998,1998,12,LEFTFIRE,0.0,10.0,"(210, 160, 3)"


In [9]:
transitions_df["action"].value_counts()

,count
action,
DOWNFIRE,275
UPLEFTFIRE,275
LEFTFIRE,275
DOWNRIGHTFIRE,265
RIGHTFIRE,245
UPFIRE,240
DOWNLEFTFIRE,220
UPRIGHTFIRE,205
